# NVIDIA NeMo Evaluator: Multi-Model Benchmarking & Comparison
### **Vishwakarma Institute of Technology, Pune** | Department of Computer Engineering
**Project Title:** NEMO Evaluator Use Case: Benchmarking and Comparing the Performance of Various Models
**Industry Collaboration:** Tata Consultancy Services (TCS)
**Team Members:** Ashish Kumar (Roll No. 110) & Aryaa Sharma (Roll No. 149)
**Internal Guide:** Prof. (Dr.) Yogesh Sharma

---
### **Overview**
This notebook enables reproducible evaluation of Large Language Models (LLMs) across standardized benchmarks:
- **MMLU-Pro** (10-choice general knowledge & multi-task reasoning)
- **GSM8K** (Multi-step mathematical reasoning)
- **MATH-500** (Challenging high-school and competition math)
- **GPQA** (Graduate-level scientific reasoning)
- **HumanEval** (Python code synthesis & unit test execution)
- **TruthfulQA** (Hallucination & misconception resistance)

In [ ]:
# Step 1: Install NVIDIA NeMo Evaluator, dependencies & CLI
!pip install -q git+https://github.com/NVIDIA-NeMo/Evaluator.git
!pip install -q pydantic pyyaml rich tabulate jinja2 scipy plotly pandas

In [ ]:
# Step 2: Configure Environment and API Credentials
import os
import getpass

# Optional: Enter your NVIDIA Build / NIM API key (Free tier available at https://build.nvidia.com)
if 'NVIDIA_API_KEY' not in os.environ:
    api_key = getpass.getpass('Enter your NVIDIA_API_KEY (leave blank to run in mock mode): ')
    if api_key.strip():
        os.environ['NVIDIA_API_KEY'] = api_key.strip()
        print('? NVIDIA NIM API Key set!')
    else:
        print('?? Running in Mock / Simulation mode (zero API cost)')

In [ ]:
# Step 3: Clone Project Repository or Download Scripts
!git clone https://github.com/ashish-vit/nemo-evaluator-benchmarking.git repo || true
%cd repo || %cd .

In [ ]:
# Step 4: Run Evaluation via CLI (nel)
!python src/cli/nel.py list
!python src/cli/nel.py eval run --config configs/eval_config_default.yaml --limit 10

In [ ]:
# Step 5: Compare Candidate vs Baseline Models
import glob
import json

runs = sorted(glob.glob('runs/*.json'), key=os.path.getmtime, reverse=True)
if len(runs) >= 2:
    run_b_id = os.path.basename(runs[0]).replace('.json', '')
    run_a_id = os.path.basename(runs[1]).replace('.json', '')
    print(f'Comparing Candidate ({run_b_id}) vs Baseline ({run_a_id}):')
    !python src/cli/nel.py compare --run-a {run_a_id} --run-b {run_b_id} --output-md colab_comparison_report.md
    !python src/cli/nel.py gate --baseline {run_a_id} --candidate {run_b_id}
else:
    print('Please execute at least 2 runs to perform comparison.')

In [ ]:
# Step 6: Interactive Radar Plot & Leaderboard in Colab
from src.storage.run_store import RunStore
import pandas as pd
import plotly.express as px

store = RunStore('runs')
lb = store.get_leaderboard()
df = pd.DataFrame(lb)
print(df[['model_name', 'best_accuracy', 'avg_latency', 'total_tokens']])

fig = px.bar(df, x='model_name', y='best_accuracy', color='model_name', title='Model Accuracy Leaderboard')
fig.show()